<a href="https://colab.research.google.com/github/thinkinginmath/ML-tutorials/blob/cursor/clip-text-to-image-search-9992/Tutorial_CLIP_Text_to_Image_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This Open in Colab badge uses the review branch `cursor/clip-text-to-image-search-9992`. It is the link that works before the pull request merges. The copy on `main` is:

https://colab.research.google.com/github/thinkinginmath/ML-tutorials/blob/main/Tutorial_CLIP_Text_to_Image_Search.ipynb

That `main` link does not open this notebook until the pull request is merged. Source on the review branch: https://github.com/thinkinginmath/ML-tutorials/blob/cursor/clip-text-to-image-search-9992/Tutorial_CLIP_Text_to_Image_Search.ipynb


# Tutorial: Text-to-Image Search with a Pretrained CLIP Model

## Learning objectives

This tutorial is for readers who can follow a small NumPy neural net and want to see what a modern vision-language model does at inference time. By the end you will be able to:

1. Explain that CLIP is already trained, and that this notebook only embeds images and text.
2. Build a small search index from public photographs with known captions.
3. Turn a sentence such as "a dog playing on the beach" into a vector, rank the index by cosine similarity, and look at the top images.
4. Say what that score is, and what it is not: it is not a probability.
5. Measure retrieval with a metric that matches the labels we actually have, and inspect a real miss.

The earlier notebook in this repository, [`Tutorial_Neural_Network_From_Scatch_(MNIST_with_NumPy_only).ipynb`](https://github.com/thinkinginmath/ML-tutorials/blob/main/Tutorial_Neural_Network_From_Scatch_(MNIST_with_NumPy_only).ipynb), trains a digit classifier from scratch. This is the computer-vision follow-up. It does not replace that notebook, and it does not ask you to retrain CLIP.

## What you need

A free Google Colab CPU runtime is enough for the default index. A GPU is optional. There is no API key, no account, and no paid model endpoint. The notebook downloads a public dataset slice and one public checkpoint the first time you run it.


## Copyright and credits

© Thinking in Math, 2026.

This tutorial text is licensed under the [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License](https://creativecommons.org/licenses/by-nc-sa/4.0/). That is the same license used by the other written tutorials in this repository.

- **Author:** thinkinginmath
- **GitHub:** https://github.com/thinkinginmath/ML-tutorials
- **YouTube:** https://www.youtube.com/channel/UC5uV1LcSkHL5zCxKBikdTEg

### Models and data this notebook uses

CLIP was introduced by Radford et al. in [Learning Transferable Visual Models From Natural Language Supervision](https://arxiv.org/abs/2103.00020) (2021). The checkpoint is OpenAI's ViT-B/32 release, hosted as [`openai/clip-vit-base-patch32`](https://huggingface.co/openai/clip-vit-base-patch32). The OpenAI CLIP code and model are under the [MIT license](https://github.com/openai/CLIP/blob/main/LICENSE). Loading is done with Hugging Face Transformers, following the current [CLIP documentation](https://huggingface.co/docs/transformers/model_doc/clip).

The photographs and captions are the COCO 2017 validation split from the [COCO Consortium](https://cocodataset.org/#home). The annotations are licensed [CC BY 4.0](https://cocodataset.org/#termsofuse). The consortium does not own the photographs. Each image keeps the Flickr license named in the annotation file, and use of the images follows the Flickr terms linked from that page. This Git repository does not contain the photographs or the model weights. Both are downloaded into the notebook runtime when you run the cells.


## How this differs from the NumPy MNIST tutorial

The MNIST notebook builds a 2-layer network and trains it. A handwritten digit goes in, ten class scores come out, and cross-entropy plus backpropagation updates the weights so that the correct digit's score grows. The labels are the digits 0 through 9, chosen before training.

This notebook starts after that kind of training is already finished. CLIP's image encoder and text encoder were pretrained together with a contrastive loss: a matching photograph and sentence are pulled toward each other, and mismatched pairs are pushed apart. We do not compute a loss, we do not run backpropagation, and we do not change a weight.

What the two notebooks share is a vector that stands for the input, and a dot product that decides the ranking. What changes is the meaning of the vector. In the MNIST net the ten outputs are class scores for a fixed label set, and softmax turns them into a distribution over those ten digits. Here an image and a sentence each become one vector in a shared space. Cosine similarity ranks candidates. A single cosine is not a distribution over classes, and softmax is only meaningful later, when you deliberately score a closed list of prompts.

Open the MNIST notebook on this repository's `main` branch if you want the training picture beside this one: [GitHub](https://github.com/thinkinginmath/ML-tutorials/blob/main/Tutorial_Neural_Network_From_Scatch_(MNIST_with_NumPy_only).ipynb), [Colab](https://colab.research.google.com/github/thinkinginmath/ML-tutorials/blob/main/Tutorial_Neural_Network_From_Scatch_(MNIST_with_NumPy_only).ipynb).


## Tutorial outline

1. Check the Python packages without replacing a working Colab PyTorch install.
2. Choose CPU or CUDA and set the size of the index.
3. Download the official COCO 2017 validation annotations, check their checksum, and keep a fixed subset of photographs.
4. Load CLIP ViT-B/32 and the preprocessor that belongs to that checkpoint.
5. Embed the photographs in batches, embed text, L2-normalize both, and rank by cosine similarity.
6. Try an editable query, then a few more queries, including negation and counting.
7. Score caption retrieval against the official image-caption links, with a chance baseline.
8. Inspect one success and one miss from that score.
9. Rank a closed list of COCO category names, and keep that exercise separate from open-ended search.


## Setup, installs, and what this run costs

Colab's current published CPU package list already includes PyTorch, Pillow, Matplotlib, NumPy, and Transformers 5. The install cell below checks versions. It installs Transformers only when the installed copy is missing or older than 5, because the CLIP helpers in Transformers 5 return a `BaseModelOutputWithPooling` and the projected vector is `pooler_output`. It installs a CPU PyTorch wheel only when PyTorch is missing. If PyTorch is present but older than 2.6, the cell stops and tells you so. It does not upgrade or remove a PyTorch that is already installed, so a Colab GPU wheel is left alone.

No token is required for Hugging Face or COCO. If the log mentions unauthenticated Hugging Face requests, you can continue. Do not create an account for this tutorial.

First-run downloads, all from public URLs, with no images or weights committed to the git repository:

| Item | What it is | Rough size |
| --- | --- | --- |
| Annotation zip | Official `annotations_trainval2017.zip`. The notebook extracts only the validation caption and instance files. | 253 MB on the wire. The checksum is checked. |
| Photographs | Default 200 images from COCO val2017, chosen with a fixed seed. | A few tens of megabytes. |
| CLIP ViT-B/32 | `openai/clip-vit-base-patch32`, about 151 million parameters, fp32 checkpoint. | About 605 MB the first time, then cached. |

On a CPU, embedding the default 200 images is a few minutes after the downloads, not an overnight job. The first run is slower because of the downloads. Later runs reuse the files under `data/coco_val2017_clip/` and the Hugging Face cache. A CUDA device is used when it exists. A paid GPU runtime is not required, and the default batch size is small on purpose.

Two environment variables exist so a short automatic check can be distinguished from the real tutorial run. Leave them unset when you are learning.

- `CLIP_TUTORIAL_MAX_IMAGES` defaults to **200**. A value of **8** is only a smoke test of the code path. Do not quote smoke-test scores as the result of the tutorial.
- `CLIP_TUTORIAL_BATCH_SIZE` defaults to **8**.

The image cap is 8 to 400. The full validation split has 5,000 images. The cap is there so a typo cannot start a much larger download. The seed that picks the subset is fixed at 0 and is not a knob for raising the score.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys


def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


def _version_tuple(version):
    parts = []
    for piece in version.split(".")[:3]:
        digits = ""
        for character in piece:
            if character.isdigit():
                digits += character
            else:
                break
        if not digits:
            break
        parts.append(int(digits))
    if not parts:
        raise RuntimeError(f"Could not parse version {version!r}.")
    return tuple(parts)


def _pip_install(arguments):
    print("Installing:", " ".join(arguments))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *arguments])


torch_version = _installed_version("torch")
if torch_version is None:
    print(
        "PyTorch is not installed. Colab already includes it. "
        "Installing a CPU wheel for this local environment only."
    )
    _pip_install(["torch", "--index-url", "https://download.pytorch.org/whl/cpu"])
    torch_version = _installed_version("torch")
if torch_version is None or _version_tuple(torch_version) < (2, 6):
    raise RuntimeError(
        f"Found PyTorch {torch_version}. Transformers 5 needs PyTorch 2.6 or newer. "
        "This notebook will not replace a PyTorch install that is already present. "
        "On Colab, use a current runtime. Outside Colab, install a current CPU wheel "
        "from https://pytorch.org/get-started/locally/ and rerun from the top."
    )

transformers_version = _installed_version("transformers")
if transformers_version is None or _version_tuple(transformers_version) < (5, 0):
    _pip_install(["transformers>=5,<6"])
    print(
        "Transformers was installed or upgraded. If an older copy was already imported "
        "in this runtime, restart the runtime and run from the top."
    )

if _installed_version("pillow") is None and importlib.util.find_spec("PIL") is None:
    _pip_install(["pillow>=10"])
if _installed_version("matplotlib") is None:
    _pip_install(["matplotlib>=3.7"])
if _installed_version("numpy") is None:
    _pip_install(["numpy>=1.26"])

print("Package check finished.")
print("torch", _installed_version("torch"))
print("transformers", _installed_version("transformers"))
print("pillow", _installed_version("pillow"))
print("matplotlib", _installed_version("matplotlib"))
print("numpy", _installed_version("numpy"))


In [ ]:
import hashlib
import json
import os
import time
import urllib.error
import urllib.request
import zipfile
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import display
from PIL import Image

%matplotlib inline

MODEL_ID = "openai/clip-vit-base-patch32"
SEED = 0
DEFAULT_MAX_IMAGES = 200
MAX_IMAGES_LIMIT = 400
DEFAULT_BATCH_SIZE = 8

ANNOTATION_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
ANNOTATION_SHA256 = "113a836d90195ee1f884e704da6304dfaaecff1f023f49b6ca93c4aaae470268"
ANNOTATION_BYTES = 252907541
COCO_VAL_URL_PREFIX = "http://images.cocodataset.org/val2017/"
EXPECTED_VAL_IMAGES = 5000
EXPECTED_CATEGORIES = 80

DATA_DIR = Path(os.environ.get("CLIP_TUTORIAL_DATA_DIR", "data/coco_val2017_clip"))
ARTIFACT_DIR = os.environ.get("CLIP_TUTORIAL_ARTIFACT_DIR", "").strip()


def read_positive_int(name, default):
    raw = os.environ.get(name)
    if raw is None or raw.strip() == "":
        return default
    try:
        value = int(raw)
    except ValueError as exc:
        raise ValueError(f"{name} must be an integer, got {raw!r}.") from exc
    if value <= 0:
        raise ValueError(f"{name} must be positive, got {value}.")
    return value


MAX_IMAGES = read_positive_int("CLIP_TUTORIAL_MAX_IMAGES", DEFAULT_MAX_IMAGES)
BATCH_SIZE = read_positive_int("CLIP_TUTORIAL_BATCH_SIZE", DEFAULT_BATCH_SIZE)
if not 8 <= MAX_IMAGES <= MAX_IMAGES_LIMIT:
    raise ValueError(
        f"CLIP_TUTORIAL_MAX_IMAGES must be from 8 to {MAX_IMAGES_LIMIT} for this tutorial. "
        f"Got {MAX_IMAGES}. The default teaching index is {DEFAULT_MAX_IMAGES}. "
        "A value of 8 is only a smoke test and is not the reported result. "
        f"COCO val2017 has {EXPECTED_VAL_IMAGES} images; the cap avoids an accidental huge download."
    )
if not 1 <= BATCH_SIZE <= 32:
    raise ValueError(f"CLIP_TUTORIAL_BATCH_SIZE must be from 1 to 32. Got {BATCH_SIZE}.")

torch.manual_seed(SEED)
np.random.seed(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU. The default index does not need a GPU.")

print(f"torch {torch.__version__} | cuda_available={torch.cuda.is_available()}")
print(f"Index size {MAX_IMAGES} | batch size {BATCH_SIZE} | seed {SEED}")
print(f"Data directory (relative unless you overrode it): {DATA_DIR}")


## Cosine similarity, before any photographs

A CLIP embedding is a list of numbers. Two lists point the same way when they describe the same thing, and different ways when they do not. The cosine of the angle between them is

$$
\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|\,\|\mathbf{v}\|}.
$$

If each vector is divided by its own length, the denominator is 1 and the cosine is just the dot product. That score lives between $-1$ and $1$. A larger score ranks higher. Nothing in that definition makes the scores add up to 1, so the score is not a probability.

The next cell is ordinary linear algebra. The same functions rank the real index later. A tie counts as a miss for Recall@1: if two images share the true image's score, the true image is not treated as the unique best hit.


In [ ]:
# CLIP_TUTORIAL_PURE_HELPERS
# Ranking math with no downloads and no model calls.
import numpy as np
import torch

def l2_normalize(embeddings, eps=1e-12):
    if not torch.is_tensor(embeddings):
        raise TypeError("embeddings must be a torch.Tensor.")
    if embeddings.ndim not in (1, 2):
        raise ValueError("embeddings must have shape (dim,) or (batch, dim).")
    if embeddings.numel() == 0 or embeddings.shape[-1] == 0:
        raise ValueError("embeddings are empty.")
    norms = torch.linalg.vector_norm(embeddings, ord=2, dim=-1, keepdim=True)
    if torch.any(norms <= eps):
        raise ValueError("Refusing to normalize a zero embedding.")
    return embeddings / norms


def assert_l2_normalized(embeddings, tol=1e-3):
    norms = torch.linalg.vector_norm(embeddings, ord=2, dim=-1)
    if not torch.allclose(norms, torch.ones_like(norms), atol=tol):
        raise RuntimeError(
            "Embeddings are not unit length "
            f"(norm range {float(norms.min()):.4f} to {float(norms.max()):.4f})."
        )


def cosine_similarity_matrix(text_embeddings, image_embeddings):
    """Dot-product of rows. Both arguments must already be L2-normalized."""
    if text_embeddings.ndim != 2 or image_embeddings.ndim != 2:
        raise ValueError("expected text shape (queries, dim) and image shape (images, dim).")
    if text_embeddings.shape[-1] != image_embeddings.shape[-1]:
        raise ValueError(
            "Text and image embeddings have different widths: "
            f"{tuple(text_embeddings.shape)} vs {tuple(image_embeddings.shape)}."
        )
    if text_embeddings.shape[0] == 0 or image_embeddings.shape[0] == 0:
        raise ValueError("Cannot score an empty query list or an empty index.")
    return text_embeddings @ image_embeddings.T


def true_pair_ranks(score_matrix):
    """0-based rank of the diagonal. An equal score on another column counts as a miss."""
    if score_matrix.ndim != 2 or score_matrix.shape[0] != score_matrix.shape[1]:
        raise ValueError(
            "Expected a square score matrix with one query caption per indexed image, "
            f"got {tuple(score_matrix.shape)}."
        )
    if score_matrix.shape[0] == 0:
        raise ValueError("Score matrix is empty.")
    true_scores = score_matrix.diag().unsqueeze(1)
    at_least_as_high = (score_matrix >= true_scores).sum(dim=1)
    return at_least_as_high - 1


def recall_at_k(ranks, k):
    if not isinstance(k, int) or isinstance(k, bool) or k < 1:
        raise ValueError(f"k must be an integer >= 1, got {k!r}.")
    ranks = torch.as_tensor(ranks)
    if ranks.numel() == 0:
        raise ValueError("No ranks to score.")
    return float((ranks < k).to(torch.float32).mean())


def chance_recall_at_k(k, n_images):
    if not isinstance(k, int) or isinstance(k, bool) or k < 1:
        raise ValueError(f"k must be an integer >= 1, got {k!r}.")
    if not isinstance(n_images, int) or isinstance(n_images, bool) or n_images < 1:
        raise ValueError(f"n_images must be an integer >= 1, got {n_images!r}.")
    return min(k, n_images) / n_images


def median_rank(ranks):
    """1-based median. An even count averages the two central ranks.

    torch.Tensor.median returns only the lower of those two, which does not
    match the chance baseline (N + 1) / 2. numpy.median uses the average.
    """
    one_based = torch.as_tensor(ranks).detach().to(dtype=torch.float64).cpu().reshape(-1) + 1
    if one_based.numel() == 0:
        raise ValueError("No ranks to summarize.")
    return float(np.median(one_based.numpy()))


def chance_median_rank(n_images):
    if not isinstance(n_images, int) or isinstance(n_images, bool) or n_images < 1:
        raise ValueError(f"n_images must be an integer >= 1, got {n_images!r}.")
    return (n_images + 1) / 2


def multilabel_recall_at_k(ranked_names, true_names, k):
    """Fraction of true labels that land in the first k ranked names."""
    if not true_names:
        raise ValueError("No ground-truth labels. Skip this image instead of scoring it as zero.")
    if not isinstance(k, int) or isinstance(k, bool) or k < 1:
        raise ValueError(f"k must be an integer >= 1, got {k!r}.")
    top = list(ranked_names)[:k]
    hits = sum(1 for name in true_names if name in top)
    return hits / len(true_names)


def choose_image_ids(image_ids, max_images, seed=0):
    ids = sorted({int(image_id) for image_id in image_ids})
    if len(ids) < max_images:
        raise RuntimeError(
            f"Need {max_images} image ids and only found {len(ids)}. "
            "Refusing to invent extra images."
        )
    generator = np.random.default_rng(seed)
    chosen = generator.choice(np.asarray(ids), size=max_images, replace=False)
    return sorted(int(image_id) for image_id in chosen)


def validate_query(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError(
            "Query text is empty. Set QUERY to a short description, "
            "for example: a dog playing on the beach."
        )
    if len(text.strip()) > 300:
        raise ValueError(
            "Query is longer than 300 characters. CLIP keeps 77 tokens, "
            "so a shorter description is the meaningful input."
        )
    return text.strip()


def validate_top_k(top_k, n_images, limit=8):
    upper = min(limit, n_images)
    if not isinstance(top_k, int) or isinstance(top_k, bool) or not 1 <= top_k <= upper:
        raise ValueError(f"TOP_K must be an integer from 1 to {upper}. Got {top_k!r}.")
    return top_k


demo = torch.tensor([[3.0, 0.0], [0.0, 4.0], [3.0, 4.0]], dtype=torch.float32)
demo_unit = l2_normalize(demo)
demo_scores = cosine_similarity_matrix(demo_unit[:1], demo_unit[1:])
print("Unit vectors:")
print(demo_unit)
print("Cosine of [3, 0] with [0, 4] and with [3, 4]:")
print(demo_scores)


## The public image collection

COCO val2017 is 5,000 everyday photographs, each with human captions (almost always five) and, in the instance file, object category names from a list of 80. People, animals, vehicles, food, and indoor and outdoor scenes are all in that collection, so a sentence about a dog, a beach, a bus, or a bowl of fruit is a fair kind of query. A 200-image slice cannot promise that any particular scene is present. If the slice has no dog on a beach, search still returns the nearest photographs, and that empty result is part of the lesson.

The annotation zip is the official file at `images.cocodataset.org`. The notebook checks the byte size and SHA-256 recorded for that URL, then extracts only `captions_val2017.json` and `instances_val2017.json`. It does not unpack the much larger training annotations. Photographs are then downloaded one by one from the `coco_url` in the caption file. A failed or unreadable file stops the notebook. Nothing is swapped in from another dataset.

The subset is `numpy`'s Generator with seed 0, then sorted. Running the default settings twice selects the same ids. Changing the seed after you have seen the score would be a different exam, so the seed stays at 0.

Redistribution limit: keep the downloaded photographs in the runtime directory for this exercise. Do not commit them, and do not assume every photograph's Flickr license allows you to republish it. The license id on each image is printed with the preview.


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def download_url(url, destination, timeout, minimum_bytes):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".part")
    last_error = None
    for attempt in range(1, 4):
        try:
            request = urllib.request.Request(
                url,
                headers={"User-Agent": "thinkinginmath-ml-tutorials-clip-notebook"},
            )
            with urllib.request.urlopen(request, timeout=timeout) as response:
                status = getattr(response, "status", 200)
                if status != 200:
                    raise RuntimeError(f"HTTP {status} for {url}")
                payload = response.read()
            if len(payload) < minimum_bytes:
                raise RuntimeError(f"Response for {url} is only {len(payload)} bytes.")
            temporary.write_bytes(payload)
            temporary.replace(destination)
            return destination
        except (urllib.error.URLError, TimeoutError, RuntimeError, OSError) as exc:
            last_error = exc
            print(f"Download attempt {attempt} failed for {url}: {exc}")
            if temporary.exists():
                temporary.unlink()
    raise RuntimeError(
        f"Could not download {url}. Last error: {last_error}. "
        "This tutorial stops instead of substituting a different dataset. "
        "Check the network and https://cocodataset.org/#download , then rerun this cell."
    ) from last_error


def load_rgb(path):
    path = Path(path)
    try:
        with Image.open(path) as image:
            rgb = image.convert("RGB")
            rgb.load()
    except Exception as exc:
        raise RuntimeError(
            f"Could not read {path.name} as an image ({exc}). "
            "Delete that file and rerun the download cell. "
            "The tutorial will not replace it with a different photo."
        ) from exc
    if min(rgb.size) < 2:
        raise RuntimeError(f"{path.name} is too small to be a usable photo: {rgb.size}.")
    return rgb


def safe_coco_file_name(file_name):
    if not isinstance(file_name, str) or Path(file_name).name != file_name:
        raise RuntimeError(f"Unexpected COCO file name: {file_name!r}.")
    if "/" in file_name or "\\" in file_name or not file_name.endswith(".jpg"):
        raise RuntimeError(f"Unexpected COCO file name: {file_name!r}.")
    return file_name


DATA_DIR.mkdir(parents=True, exist_ok=True)
zip_path = DATA_DIR / "annotations_trainval2017.zip"
download_started = time.perf_counter()
if zip_path.exists() and zip_path.stat().st_size == ANNOTATION_BYTES and sha256_file(zip_path) == ANNOTATION_SHA256:
    print("Reusing the annotation zip that already matches the official checksum.")
else:
    if zip_path.exists():
        zip_path.unlink()
    print(f"Downloading annotations ({ANNOTATION_BYTES / 1e6:.1f} MB) from {ANNOTATION_URL}")
    download_url(ANNOTATION_URL, zip_path, timeout=180, minimum_bytes=ANNOTATION_BYTES)
if zip_path.stat().st_size != ANNOTATION_BYTES or sha256_file(zip_path) != ANNOTATION_SHA256:
    raise RuntimeError(
        "The annotation zip does not match the expected size or SHA-256 "
        f"({ANNOTATION_SHA256}). Refusing to parse it. "
        "Delete the data directory and rerun. Do not point this notebook at a different dataset."
    )

caption_path = DATA_DIR / "captions_val2017.json"
instance_path = DATA_DIR / "instances_val2017.json"
wanted = {
    "annotations/captions_val2017.json": caption_path,
    "annotations/instances_val2017.json": instance_path,
}
with zipfile.ZipFile(zip_path) as archive:
    names = set(archive.namelist())
    missing = [name for name in wanted if name not in names]
    if missing:
        raise RuntimeError(f"Official annotation zip is missing {missing}.")
    for name, output_path in wanted.items():
        if name.startswith("/") or ".." in name.split("/"):
            raise RuntimeError(f"Unsafe zip member {name}.")
        output_path.write_bytes(archive.read(name))

captions_file = json.loads(caption_path.read_text(encoding="utf-8"))
instances_file = json.loads(instance_path.read_text(encoding="utf-8"))
for required in ("images", "annotations", "licenses"):
    if required not in captions_file:
        raise RuntimeError(f"captions_val2017.json has no {required!r} field.")
for required in ("images", "annotations", "categories"):
    if required not in instances_file:
        raise RuntimeError(f"instances_val2017.json has no {required!r} field.")
if len(captions_file["images"]) != EXPECTED_VAL_IMAGES:
    raise RuntimeError(
        f"Expected {EXPECTED_VAL_IMAGES} validation images in the caption file, "
        f"found {len(captions_file['images'])}."
    )
if len(instances_file["categories"]) != EXPECTED_CATEGORIES:
    raise RuntimeError(
        f"Expected {EXPECTED_CATEGORIES} COCO categories, found {len(instances_file['categories'])}."
    )

license_names = {entry["id"]: entry["name"] for entry in captions_file["licenses"]}
category_id_to_name = {}
for category in instances_file["categories"]:
    category_id_to_name[category["id"]] = category["name"]
category_names = sorted(category_id_to_name.values())
if len(category_names) != EXPECTED_CATEGORIES:
    raise RuntimeError("Category names were not unique, so the closed-set list is ambiguous.")

captions_by_image = defaultdict(list)
for annotation in captions_file["annotations"]:
    captions_by_image[annotation["image_id"]].append(annotation)
categories_by_image = defaultdict(set)
for annotation in instances_file["annotations"]:
    name = category_id_to_name.get(annotation["category_id"])
    if name is None:
        raise RuntimeError(
            f"Unknown category id {annotation['category_id']}. "
            "Refusing to drop or rename it."
        )
    categories_by_image[annotation["image_id"]].add(name)

images_by_id = {}
for image in captions_file["images"]:
    image_id = int(image["id"])
    if image_id in images_by_id:
        raise RuntimeError(f"Duplicate image id {image_id} in the caption file.")
    images_by_id[image_id] = image
instance_ids = {int(image["id"]) for image in instances_file["images"]}
if set(images_by_id) != instance_ids:
    raise RuntimeError("Caption image ids and instance image ids disagree. Refusing to continue.")

chosen_ids = choose_image_ids(images_by_id.keys(), MAX_IMAGES, seed=SEED)
image_dir = DATA_DIR / "images"
image_dir.mkdir(parents=True, exist_ok=True)
records = []
for position, image_id in enumerate(chosen_ids, start=1):
    image = images_by_id[image_id]
    file_name = safe_coco_file_name(image["file_name"])
    coco_url = image["coco_url"]
    if not coco_url.startswith(COCO_VAL_URL_PREFIX) or not coco_url.endswith(file_name):
        raise RuntimeError(f"Unexpected coco_url for image {image_id}: {coco_url}")
    caption_list = captions_by_image.get(image_id, [])
    if not caption_list:
        raise RuntimeError(f"Image {image_id} has no COCO captions.")
    primary = min(caption_list, key=lambda item: item["id"])
    query_caption = str(primary["caption"]).strip()
    if not query_caption:
        raise RuntimeError(f"Image {image_id} has an empty primary caption.")
    destination = image_dir / file_name
    if destination.exists():
        try:
            load_rgb(destination)
        except RuntimeError:
            destination.unlink()
    if not destination.exists():
        download_url(coco_url, destination, timeout=60, minimum_bytes=500)
        load_rgb(destination)
    if position == 1 or position % 25 == 0 or position == len(chosen_ids):
        print(f"Checked photograph {position}/{len(chosen_ids)}")
    records.append(
        {
            "image_id": image_id,
            "file_name": file_name,
            "path": destination,
            "coco_url": coco_url,
            "query_caption": query_caption,
            "license_name": license_names.get(image.get("license"), "unknown license id"),
            "categories": set(categories_by_image.get(image_id, set())),
            "width": int(image["width"]),
            "height": int(image["height"]),
        }
    )

if len(records) != MAX_IMAGES:
    raise RuntimeError(f"Expected {MAX_IMAGES} records and built {len(records)}.")
print(
    f"Index ready: {len(records)} photographs in {time.perf_counter() - download_started:.1f}s "
    f"(downloads plus checks). Full val split: {EXPECTED_VAL_IMAGES}."
)
print("First five image ids:", [record["image_id"] for record in records[:5]])
print("Example:", records[0]["image_id"], records[0]["width"], "x", records[0]["height"])
print("Primary caption:", records[0]["query_caption"])
print("Flickr license name recorded by COCO:", records[0]["license_name"])
print("Instance categories on that photo:", sorted(records[0]["categories"]) or "(none)")

dog_and_beach = 0
for image_id, annotations in captions_by_image.items():
    text = " ".join(str(item["caption"]).lower() for item in annotations)
    if "dog" in text and "beach" in text:
        dog_and_beach += 1
print(
    "Caption-word count on the full validation file, not a relevance label: "
    f"{dog_and_beach} images have some caption containing both 'dog' and 'beach'. "
    "The teaching index is a fixed sample, so it may contain none of those photographs."
)


In [ ]:
def short_text(text, limit=80):
    collapsed = " ".join(str(text).split())
    if len(collapsed) <= limit:
        return collapsed
    return collapsed[: limit - 1] + "…"


def save_figure(figure, file_name):
    if not ARTIFACT_DIR:
        return
    destination = Path(ARTIFACT_DIR) / file_name
    destination.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(destination, dpi=120, bbox_inches="tight")
    print(f"Saved figure {file_name}")


def show_image_row(image_paths, titles, figure_title, save_name=None):
    count = len(image_paths)
    if count == 0:
        raise ValueError("No images to display.")
    if len(titles) != count:
        raise ValueError("Need one title per image.")
    figure, axes = plt.subplots(1, count, figsize=(3.4 * count, 5.1), squeeze=False)
    for axis, path, title in zip(axes[0], image_paths, titles):
        axis.imshow(load_rgb(path))
        axis.set_title(title, fontsize=8)
        axis.axis("off")
    figure.suptitle(figure_title, fontsize=12)
    figure.tight_layout()
    save_figure(figure, save_name)
    display(figure)
    plt.close(figure)


preview = records[:6]
show_image_row(
    [record["path"] for record in preview],
    [
        f"id {record['image_id']}\n{short_text(record['query_caption'], 70)}"
        for record in preview
    ],
    "First six photographs in the fixed index, with the primary COCO caption",
    save_name="index_preview.png",
)
print("These six are the lowest image ids in the seed-0 sample, not a hand-picked gallery.")


## Load CLIP and preprocess with its own processor

The model id is `openai/clip-vit-base-patch32`: a Vision Transformer with 32×32 patches for the image, and a transformer for the text. Both were trained by OpenAI; this cell only loads the result. `CLIPProcessor` is the image preprocessor and the tokenizer shipped with that checkpoint. Resize, crop, CLIP's own mean and standard deviation, and the text tokenizer all have to be that pair. A processor from a different checkpoint would feed the model numbers it was not trained on.

`model.eval()` turns off dropout. `torch.inference_mode()` turns off autograd. Those two calls are the whole "we are not training" switch in code. There is no optimizer.

In Transformers 5, `get_image_features` and `get_text_features` return a `BaseModelOutputWithPooling`. The vector in the shared space is `pooler_output` after the projection layer. The notebook checks that this vector's width equals `config.projection_dim` (512 for this checkpoint). A vector of a different width is a different space, and ranking it would be meaningless, so the cell stops.

The forward pass of `CLIPModel` L2-normalizes those projected vectors before the dot product. We do the same normalization ourselves so the score we plot is the cosine. CLIP also multiplies the cosine by a learned temperature, `exp(logit_scale)`, when it builds classification logits. That temperature is one positive number for every pair. It changes the size of the score and it does not change which image ranks first for a single query.


In [ ]:
import transformers
from transformers import CLIPModel, CLIPProcessor
from transformers.utils import cached_file

print(f"Loading {MODEL_ID} with transformers {transformers.__version__}")
load_started = time.perf_counter()
try:
    processor = CLIPProcessor.from_pretrained(MODEL_ID)
    model = CLIPModel.from_pretrained(MODEL_ID)
except Exception as exc:
    raise RuntimeError(
        f"Could not load {MODEL_ID}. Check the network and "
        "https://huggingface.co/openai/clip-vit-base-patch32 . "
        "This notebook does not fall back to a different model. "
        "A Hugging Face token is not required."
    ) from exc

model.to(device)
model.eval()
weight_path = None
weight_name = None
for candidate in ("model.safetensors", "pytorch_model.bin"):
    try:
        weight_path = cached_file(MODEL_ID, candidate)
    except OSError:
        continue
    weight_name = candidate
    break
if weight_path is None:
    raise RuntimeError(
        f"Loaded {MODEL_ID} but could not find model.safetensors or pytorch_model.bin in the cache."
    )

parameter_count = sum(parameter.numel() for parameter in model.parameters())
logit_scale = float(model.logit_scale.detach().float().exp().cpu())
print(f"Load took {time.perf_counter() - load_started:.1f}s")
print(f"Parameters: {parameter_count:,}")
print(f"Projection width: {model.config.projection_dim}")
print(f"Checkpoint file {weight_name}: {os.path.getsize(weight_path) / 1e6:.1f} MB")
print(f"exp(logit_scale) = {logit_scale:.2f}")
print("The checkpoint filename is printed without its machine-specific cache path.")
if model.config.projection_dim != 512:
    raise RuntimeError(
        f"This tutorial expects ViT-B/32's 512-d projection, found {model.config.projection_dim}."
    )


def _pooled_embedding(features, kind):
    if not hasattr(features, "pooler_output"):
        raise RuntimeError(
            f"CLIP get_{kind}_features did not return pooler_output. "
            "Install transformers 5.x and rerun from the top. "
            "This notebook follows that documented return type and does not accept a raw tensor."
        )
    pooled = features.pooler_output
    if pooled.ndim != 2 or pooled.shape[-1] != model.config.projection_dim:
        raise RuntimeError(
            f"Expected projected {kind} embeddings of shape (batch, {model.config.projection_dim}), "
            f"got {tuple(pooled.shape)}. Refusing to rank a different vector space."
        )
    return pooled


def embed_image_paths(paths):
    if not paths:
        raise ValueError("No image paths to embed.")
    model.eval()
    pieces = []
    started = time.perf_counter()
    for start in range(0, len(paths), BATCH_SIZE):
        batch_paths = list(paths[start:start + BATCH_SIZE])
        images = [load_rgb(path) for path in batch_paths]
        encoded = processor(images=images, return_tensors="pt")
        if "pixel_values" not in encoded:
            raise RuntimeError(
                "The CLIP processor did not return pixel_values. "
                "The processor must be the one paired with this checkpoint."
            )
        pixel_values = encoded["pixel_values"].to(device)
        try:
            with torch.inference_mode():
                features = model.get_image_features(pixel_values=pixel_values, return_dict=True)
        except RuntimeError as exc:
            raise RuntimeError(
                f"Image embedding failed for a batch of {len(images)} on {device.type}. "
                "If this is an out-of-memory error, rerun with a smaller CLIP_TUTORIAL_BATCH_SIZE. "
                "The notebook does not drop images and does not switch models."
            ) from exc
        pieces.append(_pooled_embedding(features, "image").detach().to("cpu"))
        print(f"Embedded images {start + len(batch_paths)}/{len(paths)}")
    embeddings = l2_normalize(torch.cat(pieces, dim=0))
    assert_l2_normalized(embeddings)
    print(f"Image embedding took {time.perf_counter() - started:.1f}s on {device.type}.")
    return embeddings


def embed_texts(texts):
    cleaned = []
    for text in texts:
        if not isinstance(text, str) or not text.strip():
            raise ValueError("Refusing to embed an empty string.")
        cleaned.append(text.strip())
    model.eval()
    limit = int(processor.tokenizer.model_max_length)
    untruncated = processor.tokenizer(cleaned, truncation=False, add_special_tokens=True)
    too_long = sum(len(token_ids) > limit for token_ids in untruncated["input_ids"])
    if too_long:
        print(f"{too_long} text(s) are longer than {limit} tokens and will be truncated to the first {limit}.")
    encoded = processor(text=cleaned, return_tensors="pt", padding=True, truncation=True)
    allowed = {
        key: value.to(device)
        for key, value in encoded.items()
        if key in {"input_ids", "attention_mask", "position_ids"}
    }
    if "input_ids" not in allowed:
        raise RuntimeError(
            "The CLIP processor did not return input_ids. "
            "The tokenizer must be the one paired with this checkpoint."
        )
    with torch.inference_mode():
        features = model.get_text_features(**allowed, return_dict=True)
    embeddings = l2_normalize(_pooled_embedding(features, "text").detach().to("cpu"))
    assert_l2_normalized(embeddings)
    return embeddings


## Embed images in batches

Each batch is resized and normalized by the checkpoint's processor, then run through the image encoder. The batch size is 8 unless you set `CLIP_TUTORIAL_BATCH_SIZE`. Outputs are moved back to CPU and concatenated. After this cell, `image_embeddings[i]` is the unit vector for `records[i]`. Row `i` stays tied to image `i` for the rest of the notebook. That alignment is what makes the later caption score meaningful.


In [ ]:
if len(records) == 0:
    raise RuntimeError("The index is empty. Rerun the download cell. There is nothing to embed.")

image_embeddings = embed_image_paths([record["path"] for record in records])
if image_embeddings.shape != (len(records), model.config.projection_dim):
    raise RuntimeError(
        f"Image embedding matrix has shape {tuple(image_embeddings.shape)}, "
        f"expected ({len(records)}, {model.config.projection_dim})."
    )
print("Image embedding matrix:", tuple(image_embeddings.shape))


## Embed a text query and rank by cosine similarity

Edit `QUERY` and `TOP_K` in the next cell, then run that cell again. `TOP_K` has to be an integer from 1 to 8, and it cannot be larger than the index. The search embeds one sentence, normalizes it, and takes the dot product with every image vector.

The titles under the photographs are the official primary COCO caption, so you can compare your sentence with what a human wrote. A high cosine next to an unrelated caption is a miss, even if the number looks large. Read the cosine as a rank key. Do not read it as "the probability this is the right photo."


In [ ]:
# Edit these two lines, then run this cell again.
QUERY = "a dog playing on the beach"
TOP_K = 5

query_text = validate_query(QUERY)
top_k = validate_top_k(TOP_K, len(records))
query_embedding = embed_texts([query_text])
query_scores = cosine_similarity_matrix(query_embedding, image_embeddings).squeeze(0)
order = torch.argsort(query_scores, descending=True)
chosen = order[:top_k]
chosen_scores = query_scores[chosen]

show_image_row(
    [records[index]["path"] for index in chosen.tolist()],
    [
        f"cosine {query_scores[index]:.3f}\nid {records[index]['image_id']}\n"
        f"{short_text(records[index]['query_caption'], 64)}"
        for index in chosen.tolist()
    ],
    f"Top {top_k} for: {query_text}",
    save_name="query_dog_playing_on_beach.png" if query_text == "a dog playing on the beach" else "query_custom.png",
)
print("Ranked image ids:", [records[index]["image_id"] for index in chosen.tolist()])
print("Cosines:", [round(float(score), 4) for score in chosen_scores.tolist()])
print("Largest cosine in the whole index:", round(float(query_scores.max()), 4))
print("Smallest cosine in the whole index:", round(float(query_scores.min()), 4))


## Try several queries

The next cell runs five fixed sentences and shows four photographs for each. Two of them ask for ordinary COCO scenes. Two of them are there to be difficult: a negation, and a count. This gallery is not a scored benchmark. We do not have a hand-labeled relevant set for these exact sentences, and the notebook will not treat CLIP's own ranking as the ground truth.

When you look at the rows, ask a plain question. Did the words that the model seems to have used match the words you cared about? Negation ("not on a beach") and counts ("three dogs") are common weak spots for this kind of model. A photograph can score well because it contains a dog, while the "not" or the "three" is ignored.


In [ ]:
EXAMPLE_QUERIES = [
    "a person riding a bicycle",
    "a bowl of fruit on a table",
    "a red bus on a city street",
    "a dog that is not on a beach",
    "three dogs",
]
example_top_k = 4
example_embeddings = embed_texts(EXAMPLE_QUERIES)
example_scores = cosine_similarity_matrix(example_embeddings, image_embeddings)
figure, axes = plt.subplots(
    len(EXAMPLE_QUERIES),
    example_top_k,
    figsize=(3.15 * example_top_k, 3.15 * len(EXAMPLE_QUERIES)),
    squeeze=False,
)
for row, sentence in enumerate(EXAMPLE_QUERIES):
    row_order = torch.argsort(example_scores[row], descending=True)[:example_top_k]
    for column, index in enumerate(row_order.tolist()):
        axis = axes[row][column]
        axis.imshow(load_rgb(records[index]["path"]))
        axis.axis("off")
        score = float(example_scores[row, index])
        heading = f"{score:.3f}\n{short_text(records[index]['query_caption'], 42)}"
        if column == 0:
            heading = short_text(sentence, 42) + "\n" + heading
        axis.set_title(heading, fontsize=7)
figure.suptitle("Four nearest photographs for each example sentence", fontsize=13)
figure.tight_layout()
save_figure(figure, "example_queries.png")
display(figure)
plt.close(figure)
print("Example queries are qualitative. They are not given a Recall number.")


## Retrieval evaluation

The scored question is: if we hide one official caption and use it as the query, does its own photograph come back near the top of this index?

For each indexed photograph we take one caption, the annotation with the smallest id. That choice is fixed, and we do not try the other captions in order to keep the one that ranks best. The image embeddings were not trained on these captions in this notebook. There is still a limit on what the number means: CLIP's pretraining data came from the web, and COCO photographs also came from Flickr, so some of these images may have been seen in pretraining. Treat the score as a measurement of this index, not as a clean claim about generalization to new photographs, and not as a leaderboard number for all of COCO.

One relevant photograph per query makes the chance baseline simple. A random ranking puts the right photograph in the top $k$ with probability $k / N$. The median rank of a random ranking is $(N+1)/2$, where rank 1 means the right photograph is first. When the number of queries is even, that median is the average of the two central ranks, the same convention as the chance baseline. Recall@$k$ here is the fraction of queries whose right photograph has rank better than or equal to $k$ (rank 0 in the zero-based table is first place). Ties count against Recall@1.

This is caption retrieval. It is the right metric because the caption and the photograph were paired by the dataset, not by this model. It is not a score for the free-text beach query. A sentence you invent does not have an official relevant set in the file.


In [ ]:
caption_embeddings = embed_texts([record["query_caption"] for record in records])
caption_scores = cosine_similarity_matrix(caption_embeddings, image_embeddings)
ranks = true_pair_ranks(caption_scores)
evaluated = int(ranks.numel())
if evaluated != len(records):
    raise RuntimeError("Rank vector does not line up with the index.")

print(f"Caption retrieval on this index. One official caption per image. N = {evaluated}.")
print("These numbers describe this fixed subset only.")
for k in (1, 5, 10):
    measured = recall_at_k(ranks, k)
    chance = chance_recall_at_k(k, evaluated)
    print(f"Recall@{k:<2}  {measured:.3f}    chance {chance:.3f}")
print(
    f"Median rank {median_rank(ranks):.1f} (1 is a perfect hit). "
    f"Chance median {chance_median_rank(evaluated):.1f}."
)

success_order = torch.argsort(ranks, descending=False)
success_index = None
for index in success_order.tolist():
    if int(ranks[index]) == 0:
        # Prefer a clear win: the true cosine is well above the runner-up.
        row = caption_scores[index]
        best_other = torch.topk(row, k=2).values[1]
        if float(row[index] - best_other) > 0.02:
            success_index = index
            break
if success_index is None:
    for index in success_order.tolist():
        if int(ranks[index]) == 0:
            success_index = index
            break

if success_index is None:
    print("No query caption retrieved its own photograph in first place on this run.")
else:
    show_image_row(
        [records[success_index]["path"]],
        [
            f"rank 1 of {evaluated}\ncosine {float(caption_scores[success_index, success_index]):.3f}\n"
            f"{short_text(records[success_index]['query_caption'], 80)}"
        ],
        "A caption that retrieved its own photograph first",
        save_name="caption_retrieval_success.png",
    )
    print("Success image id:", records[success_index]["image_id"])
    print("Success caption:", records[success_index]["query_caption"])


## Inspect a failure

The next cell finds the official caption whose own photograph landed farthest down the list. You see the query caption, the photograph it was written for, and the photograph the model ranked first instead.

Look at the two captions and the two pictures and decide which words could have caused the mix-up. Shared objects, shared places, and similar colors are typical reasons. This is one example from this index, chosen by the rank, not a story written in advance. If every caption on a tiny smoke-test index happens to hit first place, the cell says so and shows the smallest margin instead of inventing a miss.


In [ ]:
worst_index = int(torch.argmax(ranks).item())
worst_row = caption_scores[worst_index]
top_index = int(torch.argmax(worst_row).item())
worst_rank = int(ranks[worst_index]) + 1
print(f"Worst true-image rank: {worst_rank} of {len(records)} (1 would be a hit).")
print("Query caption:", records[worst_index]["query_caption"])
print("True image id:", records[worst_index]["image_id"], "cosine", round(float(worst_row[worst_index]), 4))
print(
    "Top-ranked image id:",
    records[top_index]["image_id"],
    "cosine",
    round(float(worst_row[top_index]), 4),
)
print("Top-ranked image's own primary caption:", records[top_index]["query_caption"])
print("Categories on the true image:", sorted(records[worst_index]["categories"]) or "(none)")
print("Categories on the top-ranked image:", sorted(records[top_index]["categories"]) or "(none)")

if top_index == worst_index:
    runner_up = int(torch.topk(worst_row, k=2).indices[1].item())
    print(
        "The true photograph is still first. Showing it beside the runner-up. "
        "There is no worse miss than this on the current index."
    )
    pair = [worst_index, runner_up]
    pair_titles = [
        f"true image, still rank 1\n{short_text(records[worst_index]['query_caption'], 70)}",
        f"runner-up cosine {float(worst_row[runner_up]):.3f}\n"
        f"{short_text(records[runner_up]['query_caption'], 70)}",
    ]
else:
    pair = [worst_index, top_index]
    pair_titles = [
        f"true image, rank {worst_rank}\ncosine {float(worst_row[worst_index]):.3f}\n"
        f"{short_text(records[worst_index]['query_caption'], 70)}",
        f"model's top image\ncosine {float(worst_row[top_index]):.3f}\n"
        f"{short_text(records[top_index]['query_caption'], 70)}",
    ]
show_image_row(
    [records[index]["path"] for index in pair],
    pair_titles,
    "Failure inspection: the official caption, its photograph, and the one ranked first",
    save_name="caption_retrieval_failure.png",
)
print("Write one sentence on what the two photographs share. That is the exercise.")


## Closed-set ranking is not open-ended search

Open-ended search ranks photographs for a sentence you type. Zero-shot class ranking does something else: it freezes the photograph and ranks a list of class prompts you wrote down in advance. CLIP's paper popularized the second one. The prompt template here is `a photo of a {category}`. The template is part of the method. Another template would move the scores.

COCO's 80 category names are that closed list. The ground truth for an image is the set of category names in the instance file, which can contain more than one name, or none. Images with no instance categories are left out of the average rather than scored as failures. The metric is the fraction of those true names that appear in the top 5 prompts, averaged over images that have at least one name. A random top 5 from 80 names hits any given name with probability $5/80 = 0.0625$, so that is the chance baseline for this average.

This remains a proxy. A category box can be a small object, and the caption can be about the scene instead. A high category score does not certify the beach query, and a CLIP-chosen category is not a new label we get to treat as truth.

Softmax is the piece that turns closed-set scores into something that adds up to 1. CLIP multiplies the cosine by `exp(logit_scale)` before that softmax. The notebook prints the peaked result for one photograph so you can see it, and then does not use it as a metric. A multi-label photograph is not one exclusive class, and a softmax over category prompts says nothing about whether an open-ended search result was right.


In [ ]:
CATEGORY_TEMPLATE = "a photo of a {name}"
category_prompts = [CATEGORY_TEMPLATE.format(name=name) for name in category_names]
category_embeddings = embed_texts(category_prompts)
category_scores = cosine_similarity_matrix(category_embeddings, image_embeddings)
if category_scores.shape != (len(category_names), len(records)):
    raise RuntimeError(f"Unexpected category score shape {tuple(category_scores.shape)}.")

per_image = []
for column, record in enumerate(records):
    truth = record["categories"]
    if not truth:
        continue
    order = torch.argsort(category_scores[:, column], descending=True)
    ranked = [category_names[index] for index in order.tolist()]
    per_image.append(multilabel_recall_at_k(ranked, truth, k=5))

if not per_image:
    raise RuntimeError(
        "No indexed photograph has COCO instance categories. "
        "There is no label list to rank, and this notebook will not invent labels."
    )
category_recall = float(np.mean(per_image))
skipped = len(records) - len(per_image)
print(
    f"Closed-set proxy: mean fraction of true category names found in the top 5 prompts "
    f"= {category_recall:.3f} on {len(per_image)} images."
)
print(f"Chance baseline = {5 / EXPECTED_CATEGORIES:.3f} (5 names out of {EXPECTED_CATEGORIES}).")
print(f"Images with no instance category, excluded from this average: {skipped}.")
print("This is not the caption-retrieval Recall, and it is not a score for a free-text query.")

focus = None
for index, record in enumerate(records):
    if "dog" in record["categories"]:
        focus = index
        break
if focus is None:
    for index, record in enumerate(records):
        if record["categories"]:
            focus = index
            break
if focus is None:
    raise RuntimeError("No labeled photograph is available to draw.")

focus_order = torch.argsort(category_scores[:, focus], descending=True)
focus_top = focus_order[:8]
truth = records[focus]["categories"]
print("Photograph", records[focus]["image_id"], "instance categories:", sorted(truth))
print("Primary caption:", records[focus]["query_caption"])
print("Top category prompts by cosine:")
for index in focus_top.tolist():
    mark = "  ground truth" if category_names[index] in truth else ""
    print(f"  {float(category_scores[index, focus]):.3f}  {category_names[index]}{mark}")

scaled = category_scores[:, focus] * logit_scale
probabilities = torch.softmax(scaled, dim=0)
top_prompt = int(focus_order[0].item())
print(
    f"Softmax over the 80 prompts, after multiplying by exp(logit_scale)={logit_scale:.1f}: "
    f"P({category_names[top_prompt]!r}) = {float(probabilities[top_prompt]):.3f}."
)
print(
    "That value adds up with the other 79 prompts to 1. "
    "It is not the probability that this photograph matches an open-ended sentence, "
    "and it is not a single-label accuracy. COCO can mark several categories on one photograph."
)

figure, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].imshow(load_rgb(records[focus]["path"]))
axes[0].axis("off")
axes[0].set_title(
    f"id {records[focus]['image_id']}\n{short_text(records[focus]['query_caption'], 60)}",
    fontsize=8,
)
bar_scores = [float(category_scores[index, focus]) for index in focus_top.tolist()]
bar_labels = [category_names[index] for index in focus_top.tolist()]
bar_colors = ["#0f766e" if label in truth else "#94a3b8" for label in bar_labels]
axes[1].barh(range(len(bar_labels)), bar_scores, color=bar_colors)
axes[1].set_yticks(range(len(bar_labels)))
axes[1].set_yticklabels(bar_labels)
axes[1].invert_yaxis()
missing_labels = sorted(truth.difference(bar_labels))
xlabel = "cosine similarity"
if missing_labels:
    xlabel += "\ntrue labels outside this top 8: " + ", ".join(missing_labels)
axes[1].set_xlabel(xlabel)
axes[1].set_title("Top 8 category prompts (teal = in the COCO label set)")
figure.tight_layout()
save_figure(figure, "zeroshot_category_ranking.png")
display(figure)
plt.close(figure)


## Limitations

Cosine similarity ranks this index. It does not tell you the probability that a photograph is relevant, and the temperature-scaled softmax only becomes a distribution after you choose a closed prompt list.

The model reflects its pretraining data. The CLIP model card reports biased and denigrating associations in some zero-shot groupings of people, and it says untested deployment is out of scope. This notebook is a small public-photo exercise. It is not a search product, not a filter for people, and not a test of your private albums. Domain shift is ordinary: medical images, satellite images, drawings, and night-time phone photos are not COCO, and a COCO demo does not predict those.

Counts, negation, and fine detail are weak. "Three dogs" may retrieve one dog. "A dog that is not on a beach" may retrieve a dog on a beach because the word "dog" dominates the sentence. Text printed inside a photograph, and small objects in a crowded scene, are easy to miss. Category names can also collide with language: the COCO name "hot dog" is food, and a prompt built from that name is not a dog.

The caption metric only asks whether the paired photograph is near the top. A retrieved photograph can share a caption's words and still miss what you meant. The category metric only asks whether annotated category names rank highly under one prompt template. Neither number is an accuracy for the editable beach query. If that query's nearest photographs are the wrong scene, the index either lacks the scene or the embedding missed it. Both outcomes are visible in the figure, and neither is papered over with a made-up accuracy.

Practical photo search uses the same idea at a larger scale: embed the library, embed the query, sort by cosine or a close variant. A production system also needs an index that can grow, an evaluation set written for that library, and a decision about licenses and safety. None of that is built here. The weights stay frozen, the index is a few hundred public photographs, and the right next step if you want training itself is the NumPy MNIST notebook, where every update is visible.


## Run it again

Leave the environment variables unset and run from the top for the default 200-image lesson. To check only that the code path runs, set `CLIP_TUTORIAL_MAX_IMAGES=8` in the environment before starting the kernel, then run from the top. Quote scores only from a default run. The seed stays 0 either way, but the 8-image slice is a different and much easier exam.

The MNIST training tutorial remains [Tutorial_Neural_Network_From_Scatch_(MNIST_with_NumPy_only).ipynb](https://github.com/thinkinginmath/ML-tutorials/blob/main/Tutorial_Neural_Network_From_Scatch_(MNIST_with_NumPy_only).ipynb). This file does not modify it.
